# 🔧 05 · Lens Aberrations and Objective Lens Design
### Why a real lens never quite matches the 'perfect lens' from Module 04

**Part B · Seeing like a microscope** · Module 05 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/05_lens_aberrations_and_objective_lens_design.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib |
| **Prerequisites** | Module 04 |
| **Estimated time** | 70–85 minutes |
| **Instrumentation topics covered** | Zernike polynomials as a basis for wavefront error; defocus, spherical aberration, coma and astigmatism and their PSF signatures; the Strehl ratio and the diffraction-limited threshold; chromatic aberration and achromatic/apochromatic correction; field curvature and plan objectives |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every aberrated PSF, Strehl-ratio curve and field-curvature image here is a simulated Zernike-polynomial wavefront model; no real objective's datasheet or measured wavefront is read from a file.

## 🎯 Learning objectives

1. Represent a lens's wavefront error as a sum of Zernike polynomials, and connect specific low-order
   terms — defocus, spherical aberration, coma, astigmatism — to their classic visual signatures in
   the PSF.
2. Simulate an aberrated PSF with `simulate_psf`'s `aberrations` argument, and quantify image
   degradation with a Strehl-ratio-like peak-intensity measure as aberration magnitude increases.
3. Explain chromatic aberration as a wavelength-dependent focal length, and describe how achromatic
   and apochromatic objective designs correct it across two or three wavelengths respectively.
4. Explain field curvature as a position-dependent focus error across the field of view, and why
   "Plan" objectives exist specifically to correct it.
5. Explain the cost/complexity trade-off behind an objective's correction class, and why a lab might
   deliberately choose a less-corrected (cheaper) objective for some experiments but not others.

## 🗺️ Contents

1. Zernike polynomials: a basis for describing wavefront error
2. Classic aberrations and their PSF signatures: defocus, spherical, coma, astigmatism
3. Quantifying degradation: the Strehl ratio and the diffraction-limited threshold
4. Chromatic aberration and achromatic/apochromatic correction
5. Field curvature and plan objectives

## 🧫 The biological question

A $300 achromat objective looks perfectly fine for a quick brightfield check of a confluent dish of
cells before an experiment. The same objective, used to acquire a multi-channel fluorescence z-stack
destined for quantitative colocalization analysis, can silently introduce a systematic registration
error between channels that no amount of careful image-analysis code will catch, because it looks
exactly like real biology: two proteins that appear slightly offset from each other at the edges of
the field of view, purely because red and green light were never brought to the same focus by that
particular lens. Knowing which lens imperfections matter for which experiment — and which expensive
correction is worth paying for — starts with understanding what "aberration" actually means, in the
same Fourier-optics language Module 04 used for the ideal, aberration-free case.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
print('No GPU is needed anywhere in this notebook — everything is plain NumPy/Matplotlib.')

In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
SEED = 5
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.1; copied so this notebook runs alone) ──
import math
import numpy as np

def zernike_radial(n: int, m: int, rho: np.ndarray) -> np.ndarray:
    """The radial part R_n^m(rho) of a Zernike polynomial (n >= |m| >= 0, n - |m| even)."""
    m = abs(m)
    R = np.zeros_like(rho)
    for k in range((n - m) // 2 + 1):
        c = ((-1) ** k * math.factorial(n - k) /
             (math.factorial(k) * math.factorial((n + m) // 2 - k) * math.factorial((n - m) // 2 - k)))
        R = R + c * rho ** (n - 2 * k)
    return R


def zernike(n: int, m: int, rho: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """A real Zernike polynomial Z_n^m(rho, theta) over the unit disk (rho in [0, 1]), the
    standard basis for describing a lens's wavefront aberrations. Positive m gives the cosine
    term, negative m the sine term, m = 0 the rotationally-symmetric radial term alone."""
    R = zernike_radial(n, m, rho)
    if m >= 0:
        return R * np.cos(m * theta)
    return R * np.sin(-m * theta)


def simulate_psf(grid: int = 160, na: float = 0.9, wavelength_um: float = 0.55,
                  pixel_um: float = 0.035, aberrations: dict[tuple[int, int], float] | None = None) -> np.ndarray:
    """A diffraction-limited (or aberrated) point-spread function by Fourier optics: the
    incoherent PSF is the squared magnitude of the Fourier transform of the pupil function
    (Fraunhofer diffraction of a circular aperture of the given NA). ``aberrations`` optionally
    maps (n, m) Zernike orders to a wavefront-error coefficient in WAVES, added as phase across
    the pupil before transforming (e.g. ``{(4, 0): 0.25}`` for 0.25 waves of primary spherical
    aberration). With no aberrations this reproduces the classic Airy pattern.
    """
    freq = np.fft.fftshift(np.fft.fftfreq(grid, d=pixel_um))
    fx, fy = np.meshgrid(freq, freq)
    rho = np.hypot(fx, fy) / (na / wavelength_um)
    theta = np.arctan2(fy, fx)
    pupil = (rho <= 1.0).astype(complex)
    if aberrations:
        rho_clipped = np.clip(rho, 0, 1)
        phase = np.zeros_like(rho)
        for (n, m), coeff in aberrations.items():
            phase = phase + coeff * zernike(n, m, rho_clipped, theta)
        pupil = pupil * np.exp(2j * np.pi * phase)
    field = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(pupil)))
    psf = np.abs(field) ** 2
    return psf / psf.sum()

---
## 1 · Zernike polynomials: a basis for describing wavefront error

Module 03 used Zernike polynomials only in passing, to define the pupil's phase. Here they take
center stage. A **Zernike polynomial** $Z_n^m(\rho,\theta)$ is one member of a family of functions
defined over the unit disk ($\rho \in [0,1]$, the normalized pupil radius) that form a convenient
basis for describing *any* smooth wavefront error — deviation of the real wavefront from the perfect
sphere a flawless lens would produce. Each low-order term has a name and a recognizable physical
meaning: $Z_2^0$ is **defocus**, $Z_2^{\pm2}$ is **astigmatism**, $Z_3^{\pm1}$ is **coma**, and
$Z_4^0$ is **primary spherical aberration** — these four account for the large majority of visible
degradation in a real, moderately well-corrected objective.

In [ ]:
# ── 1.1 visualize the wavefront maps of the four classic low-order aberrations ──
size = 200
yy, xx = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing='ij')
rho = np.hypot(xx, yy)
theta = np.arctan2(yy, xx)
mask = rho <= 1.0

modes = {'Defocus  Z(2,0)': (2, 0), 'Astigmatism  Z(2,2)': (2, 2),
         'Coma  Z(3,1)': (3, 1), 'Spherical  Z(4,0)': (4, 0)}

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (label, (n, m)) in zip(axes, modes.items()):
    wavefront = zernike(n, m, np.clip(rho, 0, 1), theta)
    wavefront = np.where(mask, wavefront, np.nan)
    im = ax.imshow(wavefront, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(label, fontsize=9)
    ax.axis('off')
plt.suptitle('Wavefront error maps across the pupil for four classic aberrations (in waves)')
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each map shows how far the real wavefront leads or lags the ideal
spherical wavefront at every point across the pupil, in units of waves. Defocus and spherical
aberration are rotationally symmetric (they only depend on $\rho$); astigmatism and coma are not —
they single out a particular direction across the pupil, which is exactly why their effect on the
image looks directional rather than symmetric.

✏️ **Your turn.** Add `'Astigmatism 45°  Z(2,-2)'` (the $(2,-2)$ mode) to `modes` and re-plot.
How does its wavefront map relate to the $(2,2)$ map already shown?

---
## 2 · Classic aberrations and their PSF signatures

Each aberration deforms the PSF in a recognizable way, because `simulate_psf` applies exactly
the phase maps from Section 1 to the pupil before taking the Fourier transform. A microscopist who
learns to recognize these four signatures by eye can often diagnose *which* aberration is degrading
an image, and sometimes even which direction to adjust a correction collar or a condenser to fix it.

In [ ]:
# ── 2.1 render the PSF for each aberration at a representative magnitude ──
na = 0.85
wavelength_um = 0.55
pixel_um = 0.03
grid = 160
coeff = 0.55  # waves of wavefront error — enough to be clearly visible

psf_ideal = simulate_psf(grid=grid, na=na, wavelength_um=wavelength_um, pixel_um=pixel_um)
extent_um = grid * pixel_um / 2

fig, axes = plt.subplots(1, 5, figsize=(14, 3.2))
axes[0].imshow(psf_ideal, cmap='inferno', extent=[-extent_um, extent_um, -extent_um, extent_um])
axes[0].set_xlim(-1.2, 1.2); axes[0].set_ylim(-1.2, 1.2)
axes[0].set_title('Diffraction-limited\n(no aberration)', fontsize=9)

for ax, (label, (n, m)) in zip(axes[1:], modes.items()):
    psf_ab = simulate_psf(grid=grid, na=na, wavelength_um=wavelength_um, pixel_um=pixel_um,
                           aberrations={(n, m): coeff})
    ax.imshow(psf_ab, cmap='inferno', extent=[-extent_um, extent_um, -extent_um, extent_um])
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
    ax.set_title(f'{label}\n({coeff} waves)', fontsize=9)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

🔍 **What just happened?** Defocus spreads the energy into a broad, still-symmetric ring pattern;
astigmatism stretches the core into an ellipse (and, at a different focus setting, into a
perpendicular ellipse — not shown here, but worth exploring in the bonus); coma produces the
textbook "comet" shape, bright on one side and tailing off on the other; spherical aberration
broadens and dims the central peak while scattering substantial energy into a low, extended halo
around it. This last one is only half the real signature of spherical aberration — the halo's *size*
also changes asymmetrically above versus below true focus, something a single in-focus plane can't
show (see the Bonus section).

✏️ **Your turn.** Lower `coeff` to `0.15` (closer to a well-corrected, high-quality objective)
and re-run. Which aberration's signature is hardest to distinguish from the diffraction-limited case
at this smaller magnitude?

---
## 3 · Quantifying degradation: the Strehl ratio and the diffraction-limited threshold

Eyeballing PSF images is useful for diagnosis but not for specification. The standard
quantitative measure is the **Strehl ratio**: the ratio of the aberrated PSF's peak intensity to the
diffraction-limited PSF's peak intensity, for the same total light collected. A widely used rule of
thumb — the **Marechal criterion** — calls a system "diffraction-limited" (meaning aberrations are
small enough to be considered negligible in practice) when the Strehl ratio is at least about 0.8,
corresponding to roughly $\lambda/14$ RMS wavefront error.

In [ ]:
# ── 3.1 Strehl ratio as a function of aberration magnitude ──
def strehl_ratio(aberration, coeff, **psf_kwargs):
    psf_ref = simulate_psf(**psf_kwargs)
    psf_ab = simulate_psf(aberrations={aberration: coeff}, **psf_kwargs)
    return psf_ab.max() / psf_ref.max()

psf_kwargs = dict(grid=160, na=0.85, wavelength_um=0.55, pixel_um=0.03)
coeffs = np.linspace(0.0, 1.0, 25)

fig, ax = plt.subplots(figsize=(7, 3.5))
for label, mode in modes.items():
    strehls = [strehl_ratio(mode, c, **psf_kwargs) for c in coeffs]
    ax.plot(coeffs, strehls, marker='o', ms=3, label=label)
ax.axhline(0.8, color='gray', ls='--', label='Marechal criterion (Strehl = 0.8)')
ax.set_xlabel('wavefront error coefficient (waves)'); ax.set_ylabel('Strehl ratio')
ax.set_title('How much aberration before a lens stops being "diffraction-limited"?')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Every aberration's Strehl ratio starts at 1.0 (no error) and falls as
the coefficient grows, but the curves separate — some aberrations degrade the Strehl ratio faster
than others at the same nominal "waves of error", because of how their particular phase pattern
interacts with the circular pupil. The dashed line marks the conventional 0.8 threshold vendors use
when they advertise a lens as diffraction-limited.

✏️ **Your turn.** For the mode whose curve crosses the Strehl = 0.8 line *first* (at the smallest
coefficient), that aberration is the one this particular objective design would need to control most
tightly to stay "diffraction-limited." Identify it directly from the plot.

### 📘 Lecture notes — the Strehl ratio turns a wavefront map into one number

**Definition.** The Strehl ratio is the peak intensity of an aberrated PSF divided by the peak
intensity of the diffraction-limited PSF formed from the same total light; the Marechal criterion
treats Strehl $\ge 0.8$ (RMS wavefront error $\le \lambda/14$) as effectively diffraction-limited.

**Worked example.** Two lenses can have the same Strehl ratio from completely different underlying
aberrations — one dominated by coma, the other by spherical aberration — yet look visibly different
in a side-by-side PSF image, which is exactly why manufacturers report a Strehl number *and*, for
serious optical work, the full wavefront map.

**One memorable point.** *A single number (Strehl ratio) tells you how degraded a lens is; the
wavefront map (Section 1) tells you why — and only the second one tells you what to do about it.*

---
## 4 · Chromatic aberration and achromatic/apochromatic correction

Everything so far used a single wavelength. Real glass has a refractive index that depends on
wavelength (dispersion), so an uncorrected lens's effective focal length — and hence its focus
position — is slightly different for blue, green, and red light. Modeled simply, this shows up as a
wavelength-dependent **defocus** aberration relative to whatever single wavelength the lens happens
to be perfectly focused for. Objective lenses are corrected to counteract this by cementing together
multiple glass elements of different dispersion:

- An **achromat** (the simplest, cheapest correction) brings two wavelengths (typically a blue and a
  red) to a common focus, leaving a residual error — the "secondary spectrum" — at wavelengths in
  between.
- A **fluorite / semi-apochromat** uses a special low-dispersion glass to reduce that residual error
  further, at moderate extra cost.
- An **apochromat** (the most expensive, most heavily corrected) brings three or more wavelengths to
  a common focus, leaving only a very small residual error across the visible spectrum — essential
  for quantitative multi-channel fluorescence work where sub-pixel channel registration matters.

In [ ]:
# ── 4.1 a simplified model: residual defocus-vs-wavelength for three correction classes ──
# Illustrative functional shapes only, not measured dispersion data for any real glass or objective.
design_wavelength_nm = 550.0
wavelengths_nm = np.linspace(430, 670, 200)
delta = (wavelengths_nm - design_wavelength_nm) / 100.0  # roughly -1.2 .. +1.2

uncorrected = 0.35 * delta                         # single element: error grows ~monotonically
achromat = 0.10 * delta ** 2 - 0.03                # corrected (near-zero) at two wavelengths
apochromat = 0.02 * delta ** 2 - 0.004              # corrected at (approximately) three wavelengths

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.plot(wavelengths_nm, uncorrected, label='uncorrected singlet')
ax.plot(wavelengths_nm, achromat, label='achromat (2-color corrected)')
ax.plot(wavelengths_nm, apochromat, label='apochromat (3-color corrected)')
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('wavelength (nm)'); ax.set_ylabel('residual defocus (waves, illustrative)')
ax.set_title('Chromatic correction: how far off-focus is each wavelength?')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.2 render the visible consequence: a simulated multi-color point source, per correction class ──
def residual_defocus(curve, wavelength_nm):
    return float(np.interp(wavelength_nm, wavelengths_nm, curve))

channel_wavelengths_nm = {'blue': 460.0, 'green': 550.0, 'red': 640.0}
channel_colors = {'blue': 0, 'green': 1, 'red': 2}  # RGB channel index
curves = {'uncorrected singlet': uncorrected, 'achromat': achromat, 'apochromat': apochromat}

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (curve_name, curve) in zip(axes, curves.items()):
    rgb = np.zeros((grid, grid, 3))
    for channel, wl_nm in channel_wavelengths_nm.items():
        defocus_waves = residual_defocus(curve, wl_nm)
        psf_c = simulate_psf(grid=grid, na=0.85, wavelength_um=wl_nm / 1000.0, pixel_um=0.03,
                              aberrations={(2, 0): defocus_waves})
        rgb[:, :, channel_colors[channel]] = psf_c / psf_c.max()
    axes[list(curves).index(curve_name)].imshow(rgb[70:90, 70:90])
    ax.set_title(curve_name, fontsize=9)
    ax.axis('off')
plt.suptitle('A single on-axis point source imaged through each correction class (RGB composite)')
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the uncorrected singlet, the red, green and blue channels focus
to visibly different sizes, producing obvious color fringing around what should be a single white
point. The achromat tightens this up substantially; the apochromat tightens it further still. This is
exactly the visible artifact — color fringing, and its quantitative twin, inter-channel pixel
offset — that makes an uncorrected or lightly-corrected objective risky for multi-channel
colocalization work, even though it may look perfectly sharp in a single-channel brightfield image.

✏️ **Your turn.** Change `channel_wavelengths_nm['red']` to `700.0` (further from the design
wavelength) and re-run. Which correction class's fringing grows the most from this change?

---
## 5 · Field curvature and plan objectives

A lens that is perfectly focused at the center of its field of view is not automatically
perfectly focused everywhere else in that field — many lens designs naturally focus a flat object
onto a *curved* surface rather than a flat one, an aberration called **field curvature**. Left
uncorrected, the centre of an image can be pin-sharp while the corners are visibly out of focus, even
though the whole specimen itself was perfectly flat to begin with. **Plan** objectives (short for
"planachromat"/"plan-apochromat", etc.) add extra corrective elements specifically to flatten the
focal surface across the field, at extra cost and often some reduction in numerical aperture or
working distance compared to an equivalent non-plan design — another instance of the trade-offs
Module 03 introduced.

In [ ]:
# ── 5.1 simulate field curvature across a tiled specimen, with and without plan correction ──
tile = 60
n_tiles = 3
size_fc = tile * n_tiles
yy_t, xx_t = np.meshgrid(np.arange(size_fc), np.arange(size_fc), indexing='ij')
specimen_fc = (((yy_t % 20) < 2) | ((xx_t % 20) < 2)).astype(float)  # a fine code-generated grid target

def render_with_field_curvature(field_curvature_strength):
    out = np.zeros_like(specimen_fc)
    center = size_fc / 2
    for ty in range(n_tiles):
        for tx in range(n_tiles):
            y0, x0 = ty * tile, tx * tile
            patch = specimen_fc[y0:y0 + tile, x0:x0 + tile]
            patch_center_y, patch_center_x = y0 + tile / 2, x0 + tile / 2
            radial = np.hypot(patch_center_y - center, patch_center_x - center) / (size_fc / 2 * np.sqrt(2))
            defocus_coeff = field_curvature_strength * radial ** 2
            psf_patch = simulate_psf(grid=tile, na=0.6, wavelength_um=0.55, pixel_um=0.05,
                                      aberrations={(2, 0): defocus_coeff} if defocus_coeff > 1e-6 else None)
            blurred = np.real(np.fft.ifft2(np.fft.fft2(patch) * np.fft.fft2(np.fft.ifftshift(psf_patch))))
            out[y0:y0 + tile, x0:x0 + tile] = np.clip(blurred, 0, None)
    return out

non_plan_image = render_with_field_curvature(field_curvature_strength=1.1)
plan_image = render_with_field_curvature(field_curvature_strength=0.03)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(specimen_fc, cmap='gray'); axes[0].set_title('Flat specimen\n(a code-generated grid target)')
axes[1].imshow(non_plan_image / non_plan_image.max(), cmap='gray'); axes[1].set_title('Non-plan objective\n(sharp centre, blurred corners)')
axes[2].imshow(plan_image / plan_image.max(), cmap='gray'); axes[2].set_title('Plan objective\n(flat focus everywhere)')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** With a strong field-curvature term the centre tile stays sharp while
the four corner tiles become visibly blurred, even though every tile started from identically sharp,
flat ground truth. The plan-corrected version keeps every tile close to equally sharp. A real
field-curvature error is smooth across the field rather than tiled in discrete blocks — this
simulation trades a little realism for a much simpler, faster calculation, while keeping the
qualitative lesson intact.

✏️ **Your turn.** Set `field_curvature_strength` to an intermediate value like `0.4` and re-run.
At what point does the corner blur become something you would actually notice at a glance, versus
something you would only catch by careful inspection or quantitative sharpness measurement?

### 📘 Lecture notes — why not just always buy the most corrected objective?

**Definition.** Field curvature is a position-dependent focus error across the field of view;
"Plan" objectives add corrective elements specifically to flatten the focal surface, independent of
(and in addition to) chromatic correction class (achromat/fluorite/apochromat).

**Worked example.** A plan-apochromat objective corrects both chromatic aberration across three or
more wavelengths and field curvature simultaneously — the most expensive, most complex combination —
and is the standard choice for quantitative multi-channel fluorescence imaging across a wide field of
view, such as a tiled whole-slide scan.

**One memorable point.** *Every correction an objective adds — plan, fluorite, apochromatic — buys
back a specific, nameable artifact, at a specific, nameable cost in price and sometimes NA or working
distance; "better" is not a single axis, it is a shopping list matched against what your experiment
actually needs.*

## 📌 Key takeaways

- Zernike polynomials provide a standard basis for wavefront error; defocus, astigmatism, coma, and
  spherical aberration are the four low-order modes responsible for most visible degradation.
- The Strehl ratio collapses an aberrated PSF's quality into a single number; the conventional
  diffraction-limited threshold is Strehl $\ge 0.8$.
- Chromatic aberration is a wavelength-dependent focus error; achromats correct two wavelengths,
  apochromats correct three or more, each at increasing cost and complexity.
- Field curvature is a position-dependent focus error across the field of view; plan objectives
  correct it, independently of chromatic correction class.
- An objective's full correction class (e.g. "plan apochromat") is a bundle of independent
  corrections, each worth paying for only when the experiment actually needs it.

## ✅ Checkpoint

**1.** An objective produces a PSF with a pronounced "comet tail" that always points the same
direction across the field of view. Which aberration does this most resemble, and is it likely to be
rotationally symmetric in the pupil?

<details><summary>Answer</summary>
This is the classic signature of coma ($Z_3^{\pm1}$), as simulated in Section 2. Coma is not
rotationally symmetric — it depends on the angular pupil coordinate $\theta$, not just the radius
$\rho$ — which is exactly why its PSF signature is directional rather than symmetric like defocus or
spherical aberration.
</details>

**2.** Two objectives report the same Strehl ratio, 0.75. Does that guarantee their PSFs look the
same?

<details><summary>Answer</summary>
No. As Section 3 showed, different aberrations (or different mixtures of aberrations) can produce the
same Strehl ratio while looking visibly different — a coma-dominated PSF and a spherical-aberration-
dominated PSF at matched Strehl ratios do not resemble each other.
</details>

**3.** A lab doing single-channel brightfield screening of confluent cell monolayers, and a lab doing
quantitative three-color colocalization across a wide field of view, are choosing objectives. Which
lab has a stronger reason to pay for a plan apochromat, and why?

<details><summary>Answer</summary>
The colocalization lab. Chromatic correction (apochromat) controls inter-channel registration error,
which directly corrupts a colocalization measurement; plan correction keeps that registration
accurate across the whole field of view, not just the centre. The brightfield screening lab's task is
far more tolerant of both kinds of error, since it is not combining multiple wavelength channels
pixel-for-pixel.
</details>

## 🧠 Questions & discussion

- Adaptive optics (originally developed for astronomy) measures a wavefront's Zernike decomposition
  in real time and applies a compensating deformable mirror. Why might this be especially valuable
  for imaging deep into scattering biological tissue, beyond what a well-corrected but fixed
  objective lens can achieve?
- This module modeled field curvature as varying only with radial distance from the field centre. Are
  there other aberrations that might also vary across the field of view in a real microscope, rather
  than being fixed properties of the "on-axis" PSF this course has mostly simulated?
- If an apochromat objective already corrects chromatic aberration so well, why would a microscopy
  facility still perform routine channel-alignment calibration before a quantitative experiment?

## 🚀 Bonus — going further

- Extend Section 2's spherical-aberration PSF to a small through-focus stack (vary a small amount of
  added defocus on top of the fixed spherical-aberration term) and confirm the asymmetric
  above-focus-vs-below-focus signature that a single in-focus plane cannot show.
- Combine two aberrations at once (e.g. `{(3, 1): 0.3, (2, 2): 0.2}`) in `simulate_psf` and see
  whether the resulting PSF signature is still recognizable as a mixture of the two individual
  patterns from Section 2.
- Compute the Strehl ratio for a *combination* of small amounts of all four aberrations simultaneously
  and compare it to the sum of each aberration's individual Strehl degradation — is the combined
  effect roughly additive?

## 📝 Lab note

```text
Module: 05 — Lens Aberrations and Objective Lens Design
Date:
The correction class (achromat / fluorite / apochromat; plan / non-plan) I would specify for MY
own imaging question, and why:
One aberration signature from Section 2 I would now recognize if I saw it in a real image:
```

## 📚 References

1. Born, M., Wolf, E. (1999). *Principles of Optics* (7th ed.). Cambridge University Press.
2. Mahajan, V.N. (2001). Strehl ratio for primary aberrations in terms of their aberration variance.
   *Journal of the Optical Society of America*, 73(6), 860-861.
3. Noll, R.J. (1976). Zernike polynomials and atmospheric turbulence. *Journal of the Optical Society
   of America*, 66(3), 207-211.
4. Davidson, M.W., Abramowitz, M. Optical aberrations. In *Encyclopedia of Imaging Science and
   Technology*. Wiley.
5. Murphy, D.B., Davidson, M.W. (2012). *Fundamentals of Light Microscopy and Electronic Imaging*
   (2nd ed.). Wiley-Blackwell. — a fuller treatment of objective correction classes and specifications.

## ➡️ Next

Module 06 moves from the lens itself to how the specimen is illuminated and rendered into contrast:
bright-field, dark-field, phase contrast, and DIC each turn the same underlying optics toward making
a transparent, nearly invisible specimen visible in the first place.